# End-to-End RAG App: PDFs → Answers

**Retrieval-Augmented Generation (RAG)** lets a language model answer questions using *your*
documents instead of only what it memorised during training:

1. **Retrieve** the passages most relevant to the question.
2. **Augment** the prompt by stuffing those passages in as context.
3. **Generate** an answer grounded in that context.

I build the pipeline **one cell at a time**, running each stage on a real PDF so I can inspect the
output before moving on:

```
PDF -> text -> chunks -> embeddings -> FAISS -> retriever -> QA -> Gradio UI
```

## Step 0 — Install dependencies

- `pypdf` — read text out of PDFs
- `langchain-*` — Document objects, text splitting, the RetrievalQA chain
- `langchain-huggingface` / `sentence-transformers` — text → embedding vectors
- `transformers` — run the local generator model (phi-1_5)
- `faiss` (via `langchain-community`) — the vector index we search
- `gradio` — the web UI

In [1]:
# Run once, then restart the kernel (uncomment the line below).
# %pip install langchain langchain-community langchain-core langchain-huggingface transformers pypdf sentence-transformers gradio tf-keras
# For scanned / text-as-outline PDFs, the OCR fallback in Step 4 also needs:
# %pip install pymupdf rapidocr-onnxruntime onnxruntime numpy

## Step 1 — Force offline mode & quiet warnings

This machine can't reach the internet, so everything runs **fully offline** from the local cache:

- `HF_HUB_OFFLINE` / `TRANSFORMERS_OFFLINE` — never call out to the HuggingFace Hub.
- `GRADIO_ANALYTICS_ENABLED=False` — stop Gradio from firing telemetry over the network. On an
  offline box that background request otherwise leaves a dead HTTP client, which later surfaces as
  `RuntimeError: Cannot send a request, as the client has been closed` when you click a button.

These must all be set *before* `transformers` / `gradio` are imported, so this cell runs first.

There are two separate sources of noise: Python `warnings` (handled by `warnings.filterwarnings`)
**and** transformers' own logging (handled by `TRANSFORMERS_VERBOSITY` — `filterwarnings` does
nothing for those `[transformers] ...` lines).

In [2]:
import os
# Use only locally cached models — no network calls.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
# Disable Gradio telemetry (avoids the 'client has been closed' error when offline).
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
# transformers logs generation/tokenizer notices via its OWN logger, not the warnings module,
# so silence it here (set before transformers is imported).
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import warnings
warnings.filterwarnings('ignore')

## Step 2 — Imports

Grouped by the role each plays in the pipeline.

In [3]:
# Ingestion: read PDFs, wrap as Documents, split into chunks
from pypdf import PdfReader
import pymupdf          # render pages to images for the OCR fallback
import numpy as np      # image arrays for OCR
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vectorisation & search: embed text and store it in a FAISS index
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Generation: the local LLM, the prompt, and the RetrievalQA chain
import transformers
transformers.logging.set_verbosity_error()  # belt-and-suspenders on the env var above
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline  # current class (not the deprecated langchain_classic one)
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# UI
import gradio as gr

## Step 3 — Point at a PDF

I run the whole pipeline on a real file. Set `PDF_PATHS` to one or more PDFs on disk — everything
below reads from these variables.

In [4]:
PDF_PATHS = ["gk_ques_ans.pdf"]  # <-- change to your own PDF file path(s)
PDF_PATHS

['gk_ques_ans.pdf']

## Step 4 — PDF → text (with OCR fallback)

Ingestion: read every page of every PDF and collect the text. `extract_text()` returns `None`/empty
for pages with no *real* text — scanned images, or PDFs whose text was converted to outlines/curves
(every letter drawn as vector shapes). Those have nothing to extract.

So I use a two-tier strategy per file:

1. **Fast path** — `pypdf.extract_text()`. Works instantly for normal, text-based PDFs.
2. **OCR fallback** — if a file yields no text, render each page to an image with PyMuPDF and read
   it with **RapidOCR** (a self-contained, offline OCR engine — its models ship inside the wheel,
   nothing is downloaded). Slower, but it recovers text from scanned/outlined PDFs.

In [5]:
def ocr_pdf(path, dpi=200):
    """OCR every page of a PDF that has no extractable text (scanned / text-as-outlines)."""
    from rapidocr_onnxruntime import RapidOCR
    engine = RapidOCR()
    doc = pymupdf.open(path)
    texts = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=dpi)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        result, _ = engine(img)
        page_text = "\n".join(line[1] for line in (result or []))
        if page_text.strip():
            texts.append(page_text)
        print(f"  OCR page {i + 1}/{doc.page_count}: {len(page_text)} chars")
    return texts


pdf_texts = []
for path in PDF_PATHS:
    reader = PdfReader(path)
    extracted = [p.extract_text() for p in reader.pages]
    extracted = [t for t in extracted if t and t.strip()]

    if extracted:
        pdf_texts.extend(extracted)                       # fast path: real text found
    else:
        print(f"No embedded text in {path} -> running OCR (this can take a while)...")
        pdf_texts.extend(ocr_pdf(path))                   # fallback: scanned / outlined PDF

print(f"\nExtracted {len(pdf_texts)} page(s) of text.")

if not pdf_texts:
    raise RuntimeError(
        "Still no text after OCR. The PDF may be blank, corrupt, or in a language the OCR "
        "model doesn't cover."
    )

print("\nPreview of page 1:\n")
print(pdf_texts[0][:500])


Extracted 40 page(s) of text.

Preview of page 1:

GK Questions 
1) Which country is known as “the Land of festivals”?   
   India 
2) Where is Himalayan Mountaineering Institute located?   
   Darjeeling, India 
3) For how many disciplines is Nobel Prize awarded?   
   6 disciplines (Physics, Chemistry, Medicine, Literature, 
Economics, Peace) 
4) Name the tennis Player who is known as “The King of Clay”? 
   Rafael Nadal 
5) Which Indian City has been declared as World Heritage City (WHC) 
by UNESCO recently?        
   Ahmadabad 
6) Who wrote


## Step 5 — text → chunks

Why chunk?

1. **Retrieval quality** — smaller, focused chunks embed into more precise vectors.
2. **Context limits** — the LLM reads only a few small chunks, not a whole document.

I wrap each page as a `Document`, then split with an **overlap** so a sentence spanning a boundary
still appears intact in one chunk.

In [6]:
docs = [Document(page_content=t) for t in pdf_texts]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)
documents = splitter.split_documents(docs)

print(f"Split {len(docs)} page-document(s) into {len(documents)} chunks.")
print("\nExample chunk:\n")
print(documents[0].page_content[:400] if documents else "(no chunks)")

Split 40 page-document(s) into 79 chunks.

Example chunk:

GK Questions 
1) Which country is known as “the Land of festivals”?   
   India 
2) Where is Himalayan Mountaineering Institute located?   
   Darjeeling, India 
3) For how many disciplines is Nobel Prize awarded?   
   6 disciplines (Physics, Chemistry, Medicine, Literature, 
Economics, Peace) 
4) Name the tennis Player who is known as “The King of Clay”? 
   Rafael Nadal 
5) Which Indian City ha


## Step 6 — Load the embedding model (local, offline)

**Embeddings** map text into a vector space where semantically similar text lands close together.
There's no dedicated sentence-transformer in the local cache, so I use `distilbert-base-uncased`
(a plain BERT-family model that *is* cached). `HuggingFaceEmbeddings` wraps it with a mean-pooling
layer to turn token vectors into one 768-dim sentence vector — no download required.

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="distilbert-base-uncased"  # locally cached; loaded offline
)

# Sanity check: embed one chunk and look at the vector size
sample_vec = embedding_model.embed_query(documents[0].page_content)
print(f"Each chunk is embedded into a {len(sample_vec)}-dimensional vector.")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8664.31it/s]


Each chunk is embedded into a 768-dimensional vector.


## Step 7 — Build the FAISS index

**FAISS** is an efficient index over the chunk vectors. Given the question's vector it returns the
nearest chunk vectors — i.e. the most relevant passages. `FAISS.from_documents` embeds every chunk
and builds the index in one call. I test it with a raw similarity search below.

In [8]:
vector_store = FAISS.from_documents(documents, embedding_model)
print(f"FAISS index built over {vector_store.index.ntotal} vectors.")

# Quick retrieval test (no LLM yet) — what does the index return for a query?
hits = vector_store.similarity_search("What is this document about?", k=2)
for i, h in enumerate(hits, 1):
    print(f"\n--- Match {i} ---")
    print(h.page_content[:300])

FAISS index built over 79 vectors.

--- Match 1 ---
571) What is Optometer?        
   Instrument for testing Vision 
572) Who invented Typewriter?       
   Christopher Latham Sholes 
573) Which is the national Animal of Spain?     
   Bull 
574) How many states are there in USA?      
   50 States 
575) Where is Amar Mahal Palace located?     
   J

--- Match 2 ---
297) When is the International Day for Preservation of Ozone Layer 
Observed?          
   16th September 
298) Where is the famous Painting “Mona Lisa” Displayed?  
   Paris [Louvre Museum] 
299) Name the River, which carries Maximum Quantity of Water into 
the Sea?           
   River Amazon 
300)


## Step 8 — Prompt + LLM + retriever = QA chain

This is where retrieval and generation meet. The prompt explicitly tells the model to answer
*from the given context* — that is what makes this retrieval-**augmented** rather than the model
answering from memory. `RetrievalQA` ties the retriever, prompt, and LLM together.

For the generator I use `microsoft/phi-1_5`, a small **causal** LM that is already in the local
cache. I run it through the `text-generation` pipeline (the `text2text-generation` task was
removed in transformers 5.x, so the old FLAN-T5 recipe no longer loads). `return_full_text=False`
keeps the pipeline from echoing the prompt back.

In [9]:
# Prompt: force the model to answer from the retrieved context
prompt_template = """
Given the following information, answer the question.

Context:
{context}

Question: {question}
Answer:"""
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"],
)

# LLM: phi-1_5 is in the local cache and runs offline via the text-generation task.
generator = pipeline(
    "text-generation",
    model="microsoft/phi-1_5",
    device=-1,             # set to 0 if you have a CUDA GPU
    max_new_tokens=128,    # cap answer length
    return_full_text=False,  # return only the generated answer, not the prompt
    do_sample=False,       # deterministic (greedy) answers
    clean_up_tokenization_spaces=False,  # correct for this BPE tokenizer (avoids a warning)
)
llm = HuggingFacePipeline(pipeline=generator)

retriever = vector_store.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=False,
    chain_type_kwargs={"prompt": prompt},
)
print("QA chain ready.")

Loading weights: 100%|██████████| 341/341 [00:00<00:00, 2949.51it/s]


QA chain ready.


## Step 9 — Ask a question

Now the full pipeline runs end to end: the question is embedded, FAISS retrieves the closest
chunks, they're dropped into the prompt, and phi-1_5 generates a grounded answer.

phi-1_5 is small, so after answering it sometimes keeps generating unrelated text. I keep only
the first paragraph with a tiny `clean()` helper.

In [10]:
def clean(text):
    """Keep just the first paragraph of the model's output."""
    return text.strip().split("\n\n")[0].strip()

query = "What is this document about?"  # <-- ask anything about your PDF
# .invoke() is the current API (the old .run() is deprecated); it returns a dict.
answer = clean(qa_chain.invoke({"query": query})["result"])
print("Q:", query)
print("A:", answer)

Q: What is this document about?
A: This document is about the history of Optometry and the invention of the Typewriter.


## Step 10 — Wrap it in a Gradio UI (optional)

The cells above already prove the pipeline works. To make it interactive I add a tiny Gradio app.
A UI needs a callback, so `answer_query` is the one small function here — it just reuses the
`qa_chain` already built above.

Two small robustness touches for running inside a notebook:

- `gr.close_all()` shuts down any server left over from a previous run of this cell, so re-running
  it doesn't stack dead servers/HTTP clients.
- `analytics_enabled=False` + `prevent_thread_lock=True` keep it offline-friendly and non-blocking.

In [11]:
def answer_query(query):
    if not query or not query.strip():
        return "Please enter a valid query."
    try:
        response = clean(qa_chain.invoke({"query": query})["result"])
        return response if response and response.strip() else "No answer found."
    except Exception as e:
        return f"Error processing the query: {e}"

gr.close_all()  # tidy up any server from a previous run of this cell

with gr.Blocks(analytics_enabled=False) as demo:
    gr.Markdown("# Document QA System (RAG)")
    gr.Markdown("Ask questions about the PDF indexed above.")

    query_input = gr.Textbox(label="Your question", lines=4, placeholder="Type your question...")
    answer_output = gr.Textbox(label="Answer", lines=10, interactive=False)
    ask_btn = gr.Button("Get Answer")

    ask_btn.click(fn=answer_query, inputs=[query_input], outputs=[answer_output])

demo.launch(prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
